## 쿠텐 top10 상품 추출 및 모든 리뷰데이터 추출 코드

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import time
import random
import json
from datetime import datetime

# ── 설정 ──────────────────────────────────────────────────────────
RANK_SAVE_FILE   = "qoo10_rankings_current.jsonl"  
REVIEW_SAVE_FILE = "qoo10_reviews_master.jsonl"    


# ── 함수: 단일 상품 리뷰 전체 수집 ───────────────────────────────
def get_qoo10_reviews_all(gd_no, product_name):
    all_reviews = []
    page     = 1
    base_url = "https://www.qoo10.jp/gmkt.inc/Goods/GoodsReviewAjaxAppend.aspx"

    headers = {
        "User-Agent"     : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
        "Accept"         : "text/html, */*",
        "Accept-Language": "ja,en-US;q=0.9,en;q=0.8,ko;q=0.7",
        "Referer"        : f"https://www.qoo10.jp/item/x/{gd_no}",
        "X-Requested-With": "XMLHttpRequest",
    }

    print(f"\n  🚀 [{product_name[:35]}] 리뷰 수집 시작...")

    while True:
        params = {
            "gd_no"             : gd_no,
            "group_code"        : "2",
            "page_no"           : page,
            "page_size"         : "100",
            "sort_type"         : "P",
            "contents_cnt"      : "0",
            "___cache_expire___": int(time.time() * 1000),
        }
        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=15)

            if response.status_code != 200:
                print(f"\n  ❌ 서버 응답 에러 (코드: {response.status_code})")
                break

            html_content = response.text.strip()
            if not html_content:
                print(f"\n  ✅ 수집 완료 (총 {len(all_reviews)}개)")
                break

            soup         = BeautifulSoup(html_content, "html.parser")
            review_items = soup.find_all("li", recursive=False) or soup.select("li")

            if not review_items:
                print(f"\n  ✅ 수집 완료 (총 {len(all_reviews)}개)")
                break

            for item in review_items:
                txt_tag = item.select_one(".review_txt")
                if not txt_tag:
                    continue

                content   = txt_tag.get_text(strip=True)
                score_tag = item.select_one(".score")
                rating    = score_tag.get_text(strip=True) if score_tag else "5"
                user_info = item.select_one(".review_user_info")
                user_meta = user_info.get_text(" | ", strip=True) if user_info else ""
                type_tag  = item.select_one(".review_user_type")
                skin_info = type_tag.get_text(strip=True) if type_tag else ""

                all_reviews.append({
                    "gd_no"   : gd_no,
                    "Page"    : page,
                    "Rating"  : rating,
                    "Review"  : content,
                    "UserInfo": user_meta,
                    "SkinType": skin_info,
                })

            print(f"  🔄 {page}페이지 수집 중... (누적: {len(all_reviews)}개)", end="\r")
            page += 1
            time.sleep(1)

        except Exception as e:
            print(f"\n  ❌ 에러: {e}")
            break

    print(f"\n  ✨ 리뷰 수집 완료: {len(all_reviews)}개")
    return all_reviews


# ── STEP 1: Qoo10 Top 10 수집 (Selenium) ─────────────────────────
options = uc.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--window-size=1920,1080')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--incognito')
options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')

driver = None
try:
    driver = uc.Chrome(options=options)
    target_url = "https://www.qoo10.jp/s/?keyword_hist=k-beauty&sortType=MOST_REVIEWED&dispType=LIST&curPage=1"

    print("📡 큐텐 공식 검색 페이지 접속 중...")
    driver.get(target_url)
    time.sleep(random.uniform(5, 8))

    soup  = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.select('tr[ga-product^="goods"]')

    rank_data_list    = []
    all_review_master = []   # 전 상품 리뷰 누적
    rank_count        = 1

    for item in items:
        if rank_count > 10:
            break
        try:
            goods_code = item.get("goodscode")

            brand_tag = item.select_one("a.txt_brand")
            brand     = brand_tag.get_text(strip=True).replace("公式", "").strip() if brand_tag else "N/A"

            title_link = item.select_one(".sbj a:not(.txt_brand)")
            if not title_link:
                continue
            title       = title_link.get("title") or title_link.get_text(strip=True)
            product_url = title_link.get("href")

            price_tag     = item.select_one(".td_prc .prc strong")
            price_raw     = price_tag.get_text(strip=True) if price_tag else "0"
            price_numeric = price_raw.replace("円", "").replace(",", "").strip()

            rating_star = item.select_one(".review_rating_star")
            if rating_star and "style" in rating_star.attrs:
                width_val = rating_star["style"].split(":")[1].replace("%", "").strip()
                rating    = round(float(width_val) / 20, 2)
            else:
                rating = 0.0

            review_tag  = item.select_one(".review_total_count")
            reviews_raw = review_tag.get_text(strip=True) if review_tag else "0"
            reviews     = reviews_raw.replace("(", "").replace(")", "").replace(",", "").strip()

            rank_data_list.append({
                "rank"        : rank_count,
                "brand"       : brand,
                "title"       : title,
                "rating"      : rating,
                "reviews"     : int(reviews) if reviews.isdigit() else 0,
                "price_jpy"   : int(price_numeric) if price_numeric.isdigit() else 0,
                "url"         : product_url,
                "goods_code"  : goods_code,
                "platform"    : "Qoo10",
                "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })
            print(f"\n📍 {rank_count}위: [{brand}] {title[:35]}")

            # ── STEP 2: 전 상품 리뷰 수집 ─────────────────────────
            if goods_code:
                reviews_data = get_qoo10_reviews_all(goods_code, title)
                all_review_master.extend(reviews_data)
                print(f"  → 누적 리뷰: {len(all_review_master)}개")
            else:
                print(f"  ⚠️ goods_code 없음 — 리뷰 수집 스킵")

            rank_count += 1

        except Exception as e:
            print(f"⚠️ {rank_count}위 파싱 오류: {e}")
            continue

    # ── STEP 3: JSONL 저장 (Ulta와 동일 형식) ─────────────────────
    with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
        for entry in rank_data_list:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
        for review in all_review_master:
            f.write(json.dumps(review, ensure_ascii=False) + "\n")

    # ── STEP 4: 결과 출력 ─────────────────────────────────────────
    print()
    print("=" * 65)
    print("🏆 Qoo10 K-Beauty 베스트셀러 Top 10 (리뷰 많은 순)")
    print("=" * 65)
    for item in rank_data_list:
        print(f"{item['rank']:>2}위 | {item['brand']:<18} | {item['title'][:28]:<28} | ⭐{item['rating']} | ¥{item['price_jpy']}")
    print("=" * 65)
    print(f"\n📊 JSONL 저장 완료")
    print(f"  - 순위 파일 : {RANK_SAVE_FILE} ({len(rank_data_list)}개 상품)")
    print(f"  - 리뷰 파일 : {REVIEW_SAVE_FILE} (총 {len(all_review_master)}개 리뷰)")

except Exception as e:
    print(f"❌ 에러 발생: {e}")
finally:
    if driver:
        time.sleep(3)
        driver.quit()

📡 큐텐 공식 검색 페이지 접속 중...

📍 1위: [dear,Klairs] サプルプレパレーションフェイシャルトナー(180ml) / 韓国コスメ

  🚀 [サプルプレパレーションフェイシャルトナー(180ml) / 韓国コスメ] 리뷰 수집 시작...
  🔄 73페이지 수집 중... (누적: 7276개)
  ✅ 수집 완료 (총 7276개)

  ✨ 리뷰 수집 완료: 7276개
  → 누적 리뷰: 7276개

📍 2위: [dear,Klairs] アンセンテッドトナー(180ml) / 韓国コスメ / 化粧水 / 敏

  🚀 [アンセンテッドトナー(180ml) / 韓国コスメ / 化粧水 / 敏] 리뷰 수집 시작...
  🔄 33페이지 수집 중... (누적: 3266개)
  ✅ 수집 완료 (총 3266개)

  ✨ 리뷰 수집 완료: 3266개
  → 누적 리뷰: 10542개

📍 3위: [dear,Klairs] リッチモイストスージングクリーム(80ml) / 韓国コスメ / 水分

  🚀 [リッチモイストスージングクリーム(80ml) / 韓国コスメ / 水分] 리뷰 수집 시작...
  🔄 20페이지 수집 중... (누적: 1932개)
  ✅ 수집 완료 (총 1932개)

  ✨ 리뷰 수집 완료: 1932개
  → 누적 리뷰: 12474개

📍 4위: [ホリカホリカ] 1+1 MY FAVE PIECE EYE SHADOW //韓国コス

  🚀 [1+1 MY FAVE PIECE EYE SHADOW //韓国コス] 리뷰 수집 시작...
  🔄 19페이지 수집 중... (누적: 1880개)
  ✅ 수집 완료 (총 1880개)

  ✨ 리뷰 수집 완료: 1880개
  → 누적 리뷰: 14354개

📍 5위: [DR.GET IT] 次世代ダイエットサプリ Kスリム CLA 本格ケア 認定済み 話題のK

  🚀 [次世代ダイエットサプリ Kスリム CLA 本格ケア 認定済み 話題のK] 리뷰 수집 시작...
  🔄 18페이지 수집 중... (누적: 1708개)
  ✅ 수집 완료 (총 1708개)

  ✨ 리뷰 수집 완료: 1708